In [ ]:
import os
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.prebuilt import create_react_agent

# ---------------------------------------
# Load Environment Variables
# ---------------------------------------

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# ---------------------------------------
# Gemini LLM
# ---------------------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

# =======================================
# Tool 1 - Question Answering
# =======================================

qa_prompt = ChatPromptTemplate.from_template(
    "Answer clearly: {question}"
)

qa_chain = qa_prompt | llm


def qa_tool_func(question: str):
    result = qa_chain.invoke({"question": question})
    return result.content


qa_tool = Tool(
    name="simple_qa",
    func=qa_tool_func,
    description="Answers factual questions."
)

# =======================================
# Tool 2 - Summarizer
# =======================================

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this text:\n\n{text}"
)

summary_chain = summary_prompt | llm


def summarize(text: str):
    result = summary_chain.invoke({"text": text})
    return result.content


summary_tool = Tool(
    name="summarizer",
    func=summarize,
    description="Summarizes long text."
)

# =======================================
# Tool 3 - Tavily Search
# =======================================

search = TavilySearchResults(max_results=3)

search_tool = Tool(
    name="web_search",
    func=search.invoke,
    description="Search the web for current information."
)

# =======================================
# Create Agent
# =======================================

tools = [
    qa_tool,
    summary_tool,
    search_tool
]

agent = create_react_agent(
    model=llm,
    tools=tools
)

# =======================================
# Queries
# =======================================

queries = [
    "What is LangGraph in LangChain?",
    "Summarize: LangChain is a framework to build LLM apps using prompts, memory, tools, and agents.",
    "Latest news about Gemini AI"
]

for query in queries:

    print("\n🧑 User:", query)

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    print("\n🤖 Agent:")
    print(response["messages"][-1].content)

In [ ]:
response

In [4]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

# 🔐 Load API keys
load_dotenv(".env")
google_api_key = os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# 🔸 Initialize LLM (Gemini)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # or "gemini-1.5-pro" for higher quality
    temperature=0,
    google_api_key=google_api_key,
)

# ✅ Tool 1: Simple QA Tool
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")

@tool
def simple_qa(question: str) -> str:
    """Answers factual questions clearly."""
    chain = qa_prompt | llm
    return chain.invoke({"question": question}).content

# ✅ Tool 2: Summarizer Tool
summary_prompt = PromptTemplate.from_template("Summarize this text:\n\n{text}")

@tool
def summarizer(text: str) -> str:
    """Summarizes long paragraphs or text content."""
    chain = summary_prompt | llm
    return chain.invoke({"text": text}).content

# ✅ Tool 3: Web Search Tool (Tavily)
tavily_search = TavilySearchResults(max_results=3)

@tool
def web_search(query: str) -> str:
    """Search the internet for current and live information."""
    return tavily_search.run(query)

# 🔧 Wrap all tools in an agent (LangChain v1 create_agent API)
tools = [simple_qa, summarizer, web_search]

agent_executor = create_agent(
    model=llm,
    tools=tools,
)

# 🚀 Run user queries
queries = [
    "What is LangGraph in LangChain?"
    # "Summarize this: LangChain is a framework to build LLM apps using prompts, memory, tools, and agents.",
    # "Latest news about OpenAI GPT-4o"
]

for query in queries:
    print("\n🧑‍💻 User Query:", query)
    result = agent_executor.invoke({"messages": [{"role": "user", "content": query}]})
    response = result["messages"][-1].content
    print("\n🤖 Agent Response:", response)


🧑‍💻 User Query: What is LangGraph in LangChain?

🤖 Agent Response: LangGraph is a library built on top of LangChain that is specifically designed for building robust, stateful, and multi-actor applications with LLMs by representing their logic as a computational graph.

Here's a breakdown:

1.  **Part of LangChain:** It's an extension or specialized library within the broader LangChain ecosystem. It leverages LangChain's primitives (like LLMs, tools, chains, agents) but provides a more powerful orchestration layer.

2.  **Computational Graph:**
    *   Instead of simple linear "chains" or basic "agents" that might struggle with complex loops or state management, LangGraph allows you to define your application's flow as a **graph**.
    *   **Nodes** in the graph represent individual steps, actions, or actors (e.g., an LLM call, a tool invocation, a human review step, a specific agent).
    *   **Edges** define the transitions between these nodes, often based on the output or state of 

In [6]:
response


'LangGraph is a library built on top of LangChain that is specifically designed for building robust, stateful, and multi-actor applications with LLMs by representing their logic as a computational graph.\n\nHere\'s a breakdown:\n\n1.  **Part of LangChain:** It\'s an extension or specialized library within the broader LangChain ecosystem. It leverages LangChain\'s primitives (like LLMs, tools, chains, agents) but provides a more powerful orchestration layer.\n\n2.  **Computational Graph:**\n    *   Instead of simple linear "chains" or basic "agents" that might struggle with complex loops or state management, LangGraph allows you to define your application\'s flow as a **graph**.\n    *   **Nodes** in the graph represent individual steps, actions, or actors (e.g., an LLM call, a tool invocation, a human review step, a specific agent).\n    *   **Edges** define the transitions between these nodes, often based on the output or state of the preceding node.\n\n3.  **Stateful:**\n    *   Cruc